In [1]:
import pandas as pd
df = pd.read_csv("../data/telco_scored.csv")
df.shape

(7043, 25)

In [2]:
target = df[(df["Contract"] == "Month-to-month")
            & (df["RiskBand"] == "High")
            & (df["Churn"] == "No")]

n_target = len(target)
avg_monthly = target["MonthlyCharges"].mean()
print("Customers targeted:", n_target)
print("Average monthly charge ($):", round(avg_monthly, 2))

Customers targeted: 293
Average monthly charge ($): 83.11


In [3]:
usd_to_inr = 85          # assumed exchange rate
offer_cost_usd = 60      # cost of loyalty offer per customer (discount or perk over 12 months)
success_rate = 0.25      # share of targeted customers the offer persuades to stay
months_saved = 12        # revenue months kept per saved customer

In [4]:
# How many customers does the model expect to leave if we do nothing?
expected_churners = target["ChurnProbability"].sum()
print("Expected churners without action:", round(expected_churners))

# How many do we save?
saved = expected_churners * success_rate

# Revenue kept (in $), then in ₹
revenue_saved_usd = saved * avg_monthly * months_saved
campaign_cost_usd = n_target * offer_cost_usd

net_benefit_usd = revenue_saved_usd - campaign_cost_usd
roi_pct = net_benefit_usd / campaign_cost_usd * 100

print("Customers saved:", round(saved))
print("Revenue saved (₹):", round(revenue_saved_usd * usd_to_inr))
print("Campaign cost (₹):", round(campaign_cost_usd * usd_to_inr))
print("Net benefit (₹):", round(net_benefit_usd * usd_to_inr))
print("ROI (%):", round(roi_pct, 1))

Expected churners without action: 199
Customers saved: 50
Revenue saved (₹): 4214284
Campaign cost (₹): 1494300
Net benefit (₹): 2719984
ROI (%): 182.0


In [5]:
for rate in [0.10, 0.15, 0.25, 0.35]:
    saved = expected_churners * rate
    net = saved * avg_monthly * months_saved - n_target * offer_cost_usd
    print(f"Success rate {rate:.0%}: net benefit ₹{round(net * usd_to_inr):,}")

Success rate 10%: net benefit ₹191,414
Success rate 15%: net benefit ₹1,034,271
Success rate 25%: net benefit ₹2,719,984
Success rate 35%: net benefit ₹4,405,698


In [6]:
total_customers = len(df)
customers_needed = total_customers * 3 / 100
saved_now = expected_churners * success_rate

print("Customers to save for a 3-point drop:", round(customers_needed))
print("Saved by this campaign:", round(saved_now))
print("Churn reduction (percentage points):", round(saved_now / total_customers * 100, 2))

Customers to save for a 3-point drop: 211
Saved by this campaign: 50
Churn reduction (percentage points): 0.71


In [7]:
current = df[df["Churn"] == "No"]
m2m = current[current["Contract"] == "Month-to-month"]

groups = {
    "A: Month-to-month, High risk": m2m[m2m["RiskBand"] == "High"],
    "B: Month-to-month, Medium + High": m2m[m2m["RiskBand"].isin(["Medium", "High"])],
    "C: All current, Medium + High": current[current["RiskBand"].isin(["Medium", "High"])],
}

for name, g in groups.items():
    exp = g["ChurnProbability"].sum()
    saved = exp * success_rate
    net_usd = saved * g["MonthlyCharges"].mean() * months_saved - len(g) * offer_cost_usd
    print(f"{name}")
    print(f"  Customers: {len(g)}, expected churners: {round(exp)}, saved: {round(saved)}")
    print(f"  Net benefit: ₹{round(net_usd * usd_to_inr):,}, points reduced: {saved / total_customers * 100:.2f}")

A: Month-to-month, High risk
  Customers: 293, expected churners: 199, saved: 50
  Net benefit: ₹2,719,984, points reduced: 0.71
B: Month-to-month, Medium + High
  Customers: 1167, expected churners: 582, saved: 145
  Net benefit: ₹5,156,398, points reduced: 2.07
C: All current, Medium + High
  Customers: 1229, expected churners: 604, saved: 151
  Net benefit: ₹5,411,646, points reduced: 2.14
